# DTLN tiếng Việt: fine-tune và đánh giá

Notebook chạy trên **Google Colab GPU**, dùng checkpoint DTLN fine-tune trên VIVOS/noise để so sánh với model pretrained.

| Bước | Nội dung |
|---|---|
| 1 | Cài môi trường và kiểm tra GPU |
| 2 | Nạp mã nguồn DTLN, kết nối Google Drive |
| 3 | Tạo cặp noisy/clean từ VIVOS và noise |
| 4 | Fine-tune hoặc tiếp tục từ checkpoint đã lưu |
| 5 | Đánh giá SI-SDR/STOI, WER/CER và xem biểu đồ |

**Trước khi chạy:** chọn Runtime → Change runtime type → GPU; dữ liệu và checkpoint nằm trong MyDrive/dtln_data và MyDrive/dtln_finetune_output.

**Chỉ muốn đánh giá checkpoint đã train?** Chạy phần cài đặt, kiểm tra GPU, nạp DTLN và kết nối Drive; bỏ qua các phần tạo dữ liệu và fine-tune, rồi chạy các phần đánh giá.

## 1. Cài đặt môi trường

In [1]:
%pip -q install --no-cache-dir --force-reinstall "tensorflow[and-cuda]==2.21.0" "tf-keras==2.21.0" "numpy==2.1.3" "scipy==1.16.3" soundfile wavinfo
print('Cài xong. Nếu Colab yêu cầu restart runtime, hãy restart rồi chạy tiếp từ cell kế tiếp.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
torch 2.11.0+cu130 requires setuptools<82, but you have setuptools 84.2.1 which is incompatible.
Cài xong. Nếu Colab yêu cầu restart runtime, hãy restart rồi chạy tiếp từ cell kế tiếp.


Ô NumPy/SciPy dưới đây chỉ dùng khi gặp lỗi tương thích NumPy. Nếu chạy ô này, hãy restart runtime trước khi chạy tiếp.

In [ ]:
# Sửa riêng NumPy/SciPy nếu đã gặp lỗi _no_nep50_warning. Chạy cell này rồi restart runtime.
%pip -q install --no-cache-dir --force-reinstall "numpy==2.1.3" "scipy==1.16.3"
print('Đã cài NumPy/SciPy tương thích. Chọn Runtime > Restart session, sau đó chạy tiếp các cell import/setup.')


## 2. Kiểm tra GPU và nạp mã nguồn DTLN

In [2]:
# Phải đặt biến này trước khi import TensorFlow để dùng Keras 2 API tương thích với code DTLN cũ.
import os, sys
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np, scipy
print('NumPy:', np.__version__, '| SciPy:', scipy.__version__)
assert np.__version__ == '2.1.3', 'Hãy chạy cell sửa NumPy/SciPy, restart runtime, rồi thử lại.'
assert scipy.__version__ == '1.16.3', 'SciPy chưa đúng phiên bản; hãy chạy cell sửa NumPy/SciPy rồi restart runtime.'

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Chưa bật GPU. Chọn Runtime > Change runtime type > T4 GPU.'

NumPy: 2.1.3 | SciPy: 1.18.1 | NumPy helper: True
TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Clone mã nguồn DTLN chính thức. Có thể chạy lại cell này an toàn sau khi runtime reset.
import subprocess, pathlib
REPO = pathlib.Path('/content/DTLN')
if not (REPO / 'DTLN_model.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/breizhn/DTLN.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from DTLN_model import DTLN_model
print('DTLN source:', REPO)
print('Pretrained checkpoint:', (REPO / 'pretrained_model/model.h5').exists())

DTLN source: /content/DTLN
Pretrained checkpoint: True


## 3. Kết nối Drive và chuẩn bị dữ liệu

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Sửa đường dẫn này nếu thư mục Drive của bạn có tên khác.
DATA_ROOT = pathlib.Path('/content/drive/MyDrive/dtln_data')
VIVOS_TRAIN = DATA_ROOT / 'vivos/train/waves'
VIVOS_TEST = DATA_ROOT / 'vivos/test/waves'  # giữ riêng, không dùng trong huấn luyện
NOISE_TRAIN = DATA_ROOT / 'noise/train'
NOISE_VAL = DATA_ROOT / 'noise/val'

for p in [VIVOS_TRAIN, VIVOS_TEST, NOISE_TRAIN, NOISE_VAL]:
    print(p, 'exists=', p.exists())
assert VIVOS_TRAIN.exists() and NOISE_TRAIN.exists() and NOISE_VAL.exists(), 'Kiểm tra lại cấu trúc thư mục dữ liệu trong Drive.'
assert VIVOS_TEST.exists(), 'Cần giữ VIVOS test riêng cho đánh giá cuối.'

Mounted at /content/drive
/content/drive/MyDrive/dtln_data/vivos/train/waves exists= True
/content/drive/MyDrive/dtln_data/vivos/test/waves exists= True
/content/drive/MyDrive/dtln_data/noise/train exists= True
/content/drive/MyDrive/dtln_data/noise/val exists= True


Dữ liệu train/validation được trộn từ vivos/train và noise. VIVOS test được giữ riêng để đánh giá cuối. Ô dưới đây tạo lại các cặp audio; sau khi runtime reset, dữ liệu tạm trong /content/dtln_pairs không còn.

In [5]:
# Chuẩn bị cặp dữ liệu trên ổ đĩa tạm Colab. Validation được tách theo speaker từ VIVOS train.
# VIVOS test không được đọc ở cell này. Mỗi speaker tạo 5 bản ở SNR -5, 0, 5, 10, 15 dB.
import random, shutil, math
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd

SR = 16000
SNR_BINS = [-5, 0, 5, 10, 15]
SEED = 2026
random.seed(SEED); np.random.seed(SEED)
PREPARED = pathlib.Path('/content/dtln_pairs')
if PREPARED.exists(): shutil.rmtree(PREPARED)
for split in ['train', 'val']:
    (PREPARED / split / 'clean').mkdir(parents=True, exist_ok=True)
    (PREPARED / split / 'noisy').mkdir(parents=True, exist_ok=True)

def read_mono_16k(path):
    x, sr = sf.read(path, dtype='float32', always_2d=True)
    x = x.mean(axis=1)
    if sr != SR:
        g = gcd(int(sr), SR)
        x = resample_poly(x, SR // g, int(sr) // g).astype(np.float32)
    return x

def speech_stream(paths):
    # Nối các câu cùng người nói với khoảng lặng ngắn để tạo chuỗi đủ dài cho loader DTLN (mặc định 15s).
    gap = np.zeros(int(0.12 * SR), dtype=np.float32)
    parts = [read_mono_16k(p) for p in paths]
    return np.concatenate([part for pair in zip(parts, [gap] * len(parts)) for part in pair]).astype(np.float32)

def noise_segment(paths, n):
    path = random.choice(paths)
    x = read_mono_16k(path)
    if len(x) == 0: return np.zeros(n, dtype=np.float32)
    if len(x) < n:
        reps = int(np.ceil(n / len(x)))
        x = np.tile(x, reps)
    start = random.randint(0, len(x) - n)
    return x[start:start+n].astype(np.float32)

def mix_at_snr(clean, noise, snr_db):
    active = np.abs(clean) > max(1e-4, 0.02 * float(np.max(np.abs(clean))))
    speech_rms = np.sqrt(np.mean(clean[active] ** 2)) if np.any(active) else 1e-4
    noise_rms = np.sqrt(np.mean(noise ** 2) + 1e-12)
    noise = noise * (speech_rms / (noise_rms * (10.0 ** (snr_db / 20.0))))
    noisy = clean + noise
    peak = max(float(np.max(np.abs(noisy))), float(np.max(np.abs(clean))), 1e-6)
    scale = min(1.0, 0.98 / peak)
    return (clean * scale).astype(np.float32), (noisy * scale).astype(np.float32)

def wavs_under(folder):
    return sorted(pathlib.Path(folder).rglob('*.wav'))

speaker_dirs = [p for p in VIVOS_TRAIN.iterdir() if p.is_dir()]
if not speaker_dirs:
    raise ValueError('Không tìm thấy thư mục speaker trong vivos/train/waves. Hãy kiểm tra cấu trúc VIVOS.')
speaker_dirs = sorted(speaker_dirs)
random.shuffle(speaker_dirs)
n_val = max(1, round(0.10 * len(speaker_dirs)))
val_speakers = set(speaker_dirs[:n_val])
noise_train_files = wavs_under(NOISE_TRAIN)
noise_val_files = wavs_under(NOISE_VAL)
assert noise_train_files and noise_val_files, 'Cần WAV nhiễu trong cả noise/train và noise/val.'

counts = {'train': 0, 'val': 0}
for speaker_dir in speaker_dirs:
    split = 'val' if speaker_dir in val_speakers else 'train'
    noise_files = noise_val_files if split == 'val' else noise_train_files
    clean_files = wavs_under(speaker_dir)
    if not clean_files: continue
    clean_long = speech_stream(clean_files)
    for snr in SNR_BINS:
        clean, noisy = mix_at_snr(clean_long, noise_segment(noise_files, len(clean_long)), snr)
        stem = f'{speaker_dir.name}_snr_{snr:+03d}'
        sf.write(PREPARED / split / 'clean' / f'{stem}.wav', clean, SR, subtype='PCM_16')
        sf.write(PREPARED / split / 'noisy' / f'{stem}.wav', noisy, SR, subtype='PCM_16')
        counts[split] += 1
print('Prepared paired streams:', counts, '| validation speakers:', sorted(p.name for p in val_speakers))
print('Prepared files are in temporary Colab storage:', PREPARED)

Prepared paired streams: {'train': 205, 'val': 25} | validation speakers: ['VIVOSSPK24', 'VIVOSSPK25', 'VIVOSSPK26', 'VIVOSSPK30', 'VIVOSSPK46']
Prepared files are in temporary Colab storage: /content/dtln_pairs


## 4. Fine-tune DTLN

In [6]:
# Fine-tune pretrained DTLN. Checkpoints and logs are written to Drive each epoch.
# If Colab disconnects, rerun the notebook; this cell resumes from the newest saved epoch.
from pathlib import Path
import os, json, shutil, csv, re
from tensorflow.keras.callbacks import CSVLogger, ReduceLROnPlateau, EarlyStopping, Callback
from DTLN_model import audio_generator

TRAIN_OUTPUT = Path('/content/drive/MyDrive/dtln_finetune_output')
TRAIN_OUTPUT.mkdir(parents=True, exist_ok=True)
BASELINE_DIR = TRAIN_OUTPUT / 'baselines'
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
base_weights = REPO / 'pretrained_model/model.h5'
BASELINE_CHECKPOINT = BASELINE_DIR / 'DTLN_original_model.h5'
if not BASELINE_CHECKPOINT.exists():
    shutil.copy2(base_weights, BASELINE_CHECKPOINT)

# Keep this name unchanged when resuming. Change it only to start a new experiment.
RUN_NAME = 'DTLN_vi_snr5_lr1e-4_run01'
RUN_DIR = TRAIN_OUTPUT / f'models_{RUN_NAME}'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT = RUN_DIR / f'{RUN_NAME}.weights.h5'
TRAIN_LOG = RUN_DIR / f'training_{RUN_NAME}.log'

trainer = DTLN_model()
trainer.lr = 1e-4
trainer.batchsize = 8
trainer.max_epochs = 30
SNR_BINS = [-5, 0, 5, 10, 15]  # để cell train không phụ thuộc cell trộn dữ liệu
ALLOW_NEW_TRAINING = False  # đổi True có chủ ý khi thực sự muốn bắt đầu train mới
trainer.build_DTLN_model()
trainer.compile_model()

# Recover the latest complete epoch; if none exists, start from pretrained weights.
epoch_files = sorted(CHECKPOINT_DIR.glob('epoch_*.weights.h5'))
latest_epoch = 0
if epoch_files:
    latest_file = epoch_files[-1]
    latest_epoch = int(re.search(r'epoch_(\d+)', latest_file.name).group(1))
    trainer.model.load_weights(str(latest_file))
    print(f'Resuming from epoch {latest_epoch}: {latest_file}')
elif BEST_CHECKPOINT.exists():
    # Recovery path when epoch snapshots are missing but the saved best model exists.
    trainer.model.load_weights(str(BEST_CHECKPOINT))
    if TRAIN_LOG.exists():
        with open(TRAIN_LOG, newline='', encoding='utf-8') as f:
            log_rows = list(csv.DictReader(f))
        completed_epochs = [int(row['epoch']) + 1 for row in log_rows if row.get('epoch')]
        latest_epoch = min(trainer.max_epochs, max(completed_epochs, default=trainer.max_epochs))
    else:
        latest_epoch = trainer.max_epochs
    print(f'Loaded saved best checkpoint; latest recorded epoch={latest_epoch}.')
elif ALLOW_NEW_TRAINING:
    trainer.model.load_weights(str(BASELINE_CHECKPOINT))
    print('Starting a new run from pretrained DTLN weights by explicit request.')
else:
    raise RuntimeError('Không tìm thấy checkpoint fine-tune. Cell này không tự train lại từ pretrained. Nếu chỉ đánh giá, bỏ qua cell train; nếu chủ ý train mới, chuẩn bị lại dữ liệu và đặt ALLOW_NEW_TRAINING=True.')

with open(TRAIN_OUTPUT / f'{RUN_NAME}_config.json', 'w', encoding='utf-8') as f:
    json.dump({'base_checkpoint': str(BASELINE_CHECKPOINT), 'learning_rate': trainer.lr,
        'batch_size': trainer.batchsize, 'max_epochs': trainer.max_epochs,
        'snr_bins_db': SNR_BINS, 'run_name': RUN_NAME}, f, ensure_ascii=False, indent=2)

# Preserve best-validation tracking when resuming from an existing run.
best_val_loss = float('inf')
if TRAIN_LOG.exists():
    with open(TRAIN_LOG, newline='', encoding='utf-8') as f:
        old_rows = list(csv.DictReader(f))
    old_losses = [float(row['val_loss']) for row in old_rows if row.get('val_loss')]
    if old_losses and BEST_CHECKPOINT.exists():
        best_val_loss = min(old_losses)

class DriveCheckpoint(Callback):
    def on_epoch_end(self, epoch, logs=None):
        global best_val_loss
        logs = logs or {}
        local_tmp = Path('/content/dtln_checkpoint_tmp.weights.h5')
        epoch_path = CHECKPOINT_DIR / f'epoch_{epoch + 1:04d}.weights.h5'
        drive_tmp = Path(str(epoch_path) + '.tmp')
        self.model.save_weights(str(local_tmp))
        shutil.copy2(local_tmp, drive_tmp)
        os.replace(drive_tmp, epoch_path)
        val_loss = logs.get('val_loss')
        if val_loss is not None and float(val_loss) < best_val_loss:
            best_tmp = Path(str(BEST_CHECKPOINT) + '.tmp')
            shutil.copy2(local_tmp, best_tmp)
            os.replace(best_tmp, BEST_CHECKPOINT)
            best_val_loss = float(val_loss)
            print(f'Updated best checkpoint: val_loss={best_val_loss:.6f}')
        local_tmp.unlink(missing_ok=True)

def make_dataset(noisy_dir, clean_dir):
    length = int(np.fix(trainer.fs * trainer.len_samples / trainer.block_shift) * trainer.block_shift)
    generator = audio_generator(str(noisy_dir), str(clean_dir), length, trainer.fs)
    dataset = generator.tf_data_set.batch(trainer.batchsize, drop_remainder=True).repeat()
    steps = generator.total_samples // trainer.batchsize
    if steps < 1:
        raise ValueError(f'Không đủ audio để tạo một batch trong {noisy_dir}')
    return dataset, steps

if latest_epoch < trainer.max_epochs:
    train_ds, steps_train = make_dataset(PREPARED / 'train' / 'noisy', PREPARED / 'train' / 'clean')
    val_ds, steps_val = make_dataset(PREPARED / 'val' / 'noisy', PREPARED / 'val' / 'clean')
    callbacks = [
        DriveCheckpoint(),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-10, cooldown=1),
        CSVLogger(str(TRAIN_LOG), append=TRAIN_LOG.exists()),
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=False),
    ]
    trainer.model.fit(x=train_ds, steps_per_epoch=steps_train,
        validation_data=val_ds, validation_steps=steps_val,
        epochs=trainer.max_epochs, initial_epoch=latest_epoch,
        callbacks=callbacks, verbose=1, max_queue_size=50,
        workers=4, use_multiprocessing=True)
else:
    print(f'Training already complete through epoch {latest_epoch}; skipping training and data-pair loading.')

print('Best checkpoint:', BEST_CHECKPOINT)
print('Baseline checkpoint:', BASELINE_CHECKPOINT)
print('Training log:', TRAIN_LOG)


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, None)]               0         []                            
                                                                                                  
 lambda (Lambda)             [(None, None, 257),          0         ['input_1[0][0]']             
                              (None, None, 257)]                                                  
                                                                                                  
 lstm (LSTM)                 (None, None, 128)            197632    ['lambda[0][0]']              
                                                                                                  
 dropout (Dropout)           (None, None, 128)            0         ['lstm[0][0]']            

None


Resuming from epoch 15: /content/drive/MyDrive/dtln_finetune_output/models_DTLN_vi_snr5_lr1e-4_run01/checkpoints/epoch_0015.weights.h5
Epoch 16/30


1985/1985 [==============================] - 670s 331ms/step - loss: -15.0902 - val_loss: -12.5569 - lr: 0.0010
Epoch 17/30
1985/1985 [==============================] - 653s 329ms/step - loss: -15.2000 - val_loss: -12.5678 - lr: 0.0010
Epoch 18/30
1985/1985 [==============================] - 656s 330ms/step - loss: -15.2820 - val_loss: -12.2085 - lr: 0.0010
Epoch 19/30
1985/1985 [==============================] - 666s 336ms/step - loss: -15.3357 - val_loss: -12.4045 - lr: 0.0010
Epoch 20/30
1985/1985 [==============================] - 663s 334ms/step - loss: -15.3800 - val_loss: -12.1395 - lr: 0.0010
Epoch 21/30
1985/1985 [==============================] - 692s 349ms/step - loss: -15.4956 - val_loss: -12.7180 - lr: 5.0000e-04
Epoch 22/30
1985/1985 [==============================] - 652s 328ms/step - loss: -15.6000 - val_loss: -12.6041 - lr: 5.0000e-04
Epoch 23/30
1985/1985 [==============================] - 692s 349ms/step - loss: -15.6431 - val_loss: -12.4022 - lr: 5.0000e-04
Epoch 24

## Lưu ý trước khi dùng checkpoint

- Giữ nguyên `RUN_NAME` để notebook tự nối tiếp từ checkpoint epoch mới nhất trong `models_<RUN_NAME>/checkpoints/`; đổi tên khi bắt đầu thí nghiệm mới.
- Mỗi epoch lưu một checkpoint đầy đủ lên Drive; checkpoint tốt nhất và CSV log cũng ở trong `MyDrive/dtln_finetune_output/models_<RUN_NAME>/`.
- Nếu runtime bị ngắt, mở lại notebook và chạy lại từ đầu (bao gồm tạo cặp audio); cell fine-tune tự nạp epoch hoàn tất gần nhất. Trọng số được khôi phục, còn trạng thái optimizer/lịch giảm learning rate khởi động lại.
- Với Pro, giữ tab Colab đang kết nối và tránh cho máy tính vào sleep. Checkpoint trên Drive bảo vệ tiến độ đã lưu nếu phiên bị dừng; chúng không giữ tiến trình chạy khi runtime đã ngắt.
- Bản pretrained gốc được giữ ở `MyDrive/dtln_finetune_output/baselines/DTLN_original_model.h5`.
- Notebook chưa export ONNX. Script export DTLN gốc dùng `keras2onnx` đời cũ; hãy kiểm tra riêng trong môi trường tương thích trước khi thay model đang chạy.
- VIVOS test không bị notebook đọc khi train. Dành nó cho đánh giá cuối, tạo noisy pairs bằng noise/test độc lập rồi đo PESQ/STOI/SI-SDR và ASR trên cùng các file.
- Nếu validation loss tăng hoặc nghe thấy mất phụ âm, giảm learning rate/epoch và so sánh với checkpoint gốc trên cùng tập test.



## 5. Đánh giá khử nhiễu: SI-SDR và STOI

## Đánh giá mô hình sau fine-tune (VIVOS test)

Phần này chỉ dùng `vivos/test` để đánh giá, không cập nhật trọng số. Nó trộn cùng một nhiễu với từng câu sạch ở SNR -5/0/5/10/15 dB, rồi so sánh noisy, DTLN pretrained và checkpoint fine-tuned bằng SI-SDR và STOI. Kết quả được nhóm theo SNR, lưu CSV/PNG lên Drive và hiển thị biểu đồ cùng spectrogram mẫu.

**Lưu ý về độ độc lập:** Drive hiện có `noise/train` và `noise/val`, chưa thấy `noise/test`. Cell dưới đây ưu tiên `noise/test`; nếu chưa có thì dùng `noise/val` và in cảnh báo. Khi báo cáo chính thức, nên bổ sung bộ noise test chưa dùng ở train/validation.


**Kiểm tra cấu hình train:** file log trong Drive bắt đầu với `lr=0.001`, trong khi file config ghi `learning_rate=0.0001`. Đây là điểm chưa khớp cần xác minh trước khi ghi báo cáo về learning rate. Cell đánh giá sẽ in lại hai giá trị để bạn thấy rõ.


Phần so sánh bổ sung gồm SI-SDR gain có khoảng tin cậy, phân bố gain từng câu, scatter ghép cặp, waveform/spectrogram cùng ví dụ âm thanh. Cell ASR tùy chọn sử dụng transcript test của VIVOS và PhoWhisper-small để tính WER/CER.

In [ ]:
# Nếu thiếu STOI, cài một lần trong Colab rồi chạy tiếp cell này.
%pip -q install pystoi

from pathlib import Path
from functools import lru_cache
import random, json
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from scipy.signal import resample_poly, spectrogram
from math import gcd
from pystoi import stoi
from IPython.display import Audio, display

# Evaluation paths. This cell can run after setup/import/Drive cells; no retraining is needed.
DATA_ROOT = Path('/content/drive/MyDrive/dtln_data')
VIVOS_TEST = DATA_ROOT / 'vivos/test/waves'
NOISE_VAL = DATA_ROOT / 'noise/val'
TRAIN_OUTPUT = Path('/content/drive/MyDrive/dtln_finetune_output')
RUN_NAME = 'DTLN_vi_snr5_lr1e-4_run01'
RUN_DIR = TRAIN_OUTPUT / f'models_{RUN_NAME}'
BASELINE_CHECKPOINT = TRAIN_OUTPUT / 'baselines/DTLN_original_model.h5'

SR = 16000
EVAL_SNRS = [-5, 0, 5, 10, 15]
MAX_EVAL_FILES = 100  # đặt None để chạy toàn bộ VIVOS test
EVAL_SEED = 20261105
EVAL_OUT = TRAIN_OUTPUT / 'evaluation'
EVAL_OUT.mkdir(parents=True, exist_ok=True)
TRAIN_LOG = RUN_DIR / f'training_{RUN_NAME}.log'
CONFIG_PATH = TRAIN_OUTPUT / f'{RUN_NAME}_config.json'
if TRAIN_LOG.exists() and CONFIG_PATH.exists():
    log_preview = pd.read_csv(TRAIN_LOG)
    if len(log_preview):
        first_lr = float(log_preview['lr'].iloc[0]) if 'lr' in log_preview else float('nan')
        configured_lr = float(json.loads(CONFIG_PATH.read_text())['learning_rate'])
        print('LR ghi trong config:', configured_lr)
        print('LR ở epoch đầu trong training log:', first_lr)
        if abs(first_lr - configured_lr) > 1e-8:
            print('CẢNH BÁO: LR trong log không khớp config; đánh giá vẫn dùng đúng checkpoint đã lưu.')
FINE_WEIGHTS = RUN_DIR / f'{RUN_NAME}.weights.h5'
BASE_WEIGHTS = BASELINE_CHECKPOINT
assert FINE_WEIGHTS.exists(), f'Không thấy checkpoint fine-tune: {FINE_WEIGHTS}'
assert BASE_WEIGHTS.exists(), f'Không thấy checkpoint pretrained: {BASE_WEIGHTS}'

# noise/test là lựa chọn ưu tiên. Fallback noise/val chỉ để chạy thử, cần ghi chú khi báo cáo.
EVAL_NOISE_DIR = DATA_ROOT / 'noise/test'
NOISE_IS_HELD_OUT = EVAL_NOISE_DIR.exists()
if not NOISE_IS_HELD_OUT:
    EVAL_NOISE_DIR = NOISE_VAL
    print('CẢNH BÁO: không có noise/test; đang dùng noise/val. Nên tạo noise test độc lập cho báo cáo cuối.')
else:
    print('Dùng noise test độc lập:', EVAL_NOISE_DIR)

def eval_wavs(folder):
    return sorted(Path(folder).rglob('*.wav'))

test_files = eval_wavs(VIVOS_TEST)
noise_files = eval_wavs(EVAL_NOISE_DIR)
assert test_files, f'Không tìm thấy WAV test tại {VIVOS_TEST}'
assert noise_files, f'Không tìm thấy WAV nhiễu tại {EVAL_NOISE_DIR}'
if MAX_EVAL_FILES is not None and len(test_files) > MAX_EVAL_FILES:
    test_files = random.Random(EVAL_SEED).sample(test_files, MAX_EVAL_FILES)
print(f'{len(test_files)} câu VIVOS test × {len(EVAL_SNRS)} mức SNR; noise WAV={len(noise_files)}')

def read_audio_16k(file_path):
    x, sr = sf.read(str(file_path), dtype='float32', always_2d=True)
    x = x.mean(axis=1)
    if sr != SR:
        g = gcd(int(sr), SR)
        x = resample_poly(x, SR // g, int(sr) // g).astype(np.float32)
    return x

@lru_cache(maxsize=128)
def cached_noise(file_path):
    return read_audio_16k(file_path)

def make_noise(n, rng):
    x = cached_noise(str(rng.choice(noise_files)))
    if len(x) == 0:
        return np.zeros(n, np.float32)
    if len(x) < n:
        x = np.tile(x, int(np.ceil(n / len(x))))
    start = int(rng.integers(0, len(x) - n + 1))
    return x[start:start+n].astype(np.float32)

def mix_eval(clean, noise, snr_db):
    peak_clean = float(np.max(np.abs(clean))) if len(clean) else 0.0
    active = np.abs(clean) > max(1e-4, 0.02 * peak_clean)
    speech_rms = np.sqrt(np.mean(clean[active] ** 2)) if np.any(active) else 1e-4
    noise_rms = np.sqrt(np.mean(noise ** 2) + 1e-12)
    noise = noise * (speech_rms / (noise_rms * (10.0 ** (snr_db / 20.0))))
    noisy = clean + noise
    peak = max(float(np.max(np.abs(noisy))), peak_clean, 1e-6)
    scale = min(1.0, 0.98 / peak)
    return (clean * scale).astype(np.float32), (noisy * scale).astype(np.float32)

def si_sdr(reference, estimate):
    n = min(len(reference), len(estimate))
    ref = np.asarray(reference[:n], np.float64)
    est = np.asarray(estimate[:n], np.float64)
    ref -= ref.mean(); est -= est.mean()
    denom = np.dot(ref, ref) + 1e-12
    target = (np.dot(est, ref) / denom) * ref
    residual = est - target
    return float(10 * np.log10((np.dot(target, target) + 1e-12) / (np.dot(residual, residual) + 1e-12)))

def enhance_one(model, audio):
    # Pad 384 samples at both ends per the official DTLN evaluation routine.
    padded = np.pad(np.asarray(audio, np.float32), (384, 384))
    pred = model.predict_on_batch(padded[None, :].astype(np.float32))
    pred = np.asarray(pred).squeeze()
    return pred[384:384 + len(audio)].astype(np.float32)

def load_dtln(weights):
    obj = DTLN_model()
    obj.build_DTLN_model()
    obj.model.load_weights(str(weights))
    return obj.model

print('Loading baseline and fine-tuned DTLN weights...')
baseline_model = load_dtln(BASE_WEIGHTS)
finetuned_model = load_dtln(FINE_WEIGHTS)

rng = np.random.default_rng(EVAL_SEED)
rows = []
example = None
for file_idx, file_path in enumerate(test_files, start=1):
    clean_raw = read_audio_16k(file_path)
    if len(clean_raw) < int(0.25 * SR):
        continue
    for snr_db in EVAL_SNRS:
        clean, noisy = mix_eval(clean_raw, make_noise(len(clean_raw), rng), snr_db)
        out_base = enhance_one(baseline_model, noisy)
        out_fine = enhance_one(finetuned_model, noisy)
        for name, estimate in [('noisy', noisy), ('pretrained', out_base), ('finetuned', out_fine)]:
            rows.append({
                'file': file_path.name, 'speaker': file_path.parent.name,
                'snr_db': snr_db, 'system': name,
                'si_sdr_db': si_sdr(clean, estimate),
                'stoi': float(stoi(clean, estimate, SR, extended=False)),
            })
        if example is None and snr_db == 0:
            example = {'file': file_path.name, 'clean': clean, 'noisy': noisy,
                       'pretrained': out_base, 'finetuned': out_fine}
    if file_idx % 10 == 0 or file_idx == len(test_files):
        print(f'Processed {file_idx}/{len(test_files)} utterances')

results = pd.DataFrame(rows)
results.to_csv(EVAL_OUT / 'utterance_metrics.csv', index=False)
summary = (results.groupby(['snr_db', 'system'])[['si_sdr_db', 'stoi']]
          .mean().reset_index().sort_values(['snr_db', 'system']))
summary.to_csv(EVAL_OUT / 'summary_by_snr.csv', index=False)
display(summary.pivot(index='snr_db', columns='system', values='si_sdr_db').round(3))
display(summary.pivot(index='snr_db', columns='system', values='stoi').round(4))
print('CSV saved:', EVAL_OUT)


## 6. Đánh giá nhận dạng tiếng Việt: PhoWhisper, WER và CER

## Đánh giá tác động tới nhận dạng tiếng Việt (WER/CER)

Cell này dùng transcript chuẩn trong `vivos/test/prompts.txt` và PhoWhisper-small để so sánh nhận dạng trên cùng một nhóm câu/noise: noisy, DTLN pretrained và fine-tuned. Mặc định lấy 20 câu và SNR 0/5/10 dB để kiểm tra nhanh; tăng `ASR_MAX_FILES` trước khi chạy để làm báo cáo lớn hơn. WER/CER chỉ so sánh hợp lệ khi chuẩn hóa văn bản giống nhau cho cả ba đầu vào.

In [ ]:
%pip -q install transformers accelerate jiwer

import re, unicodedata
import torch
import jiwer
from transformers import pipeline

PROMPTS = DATA_ROOT / 'vivos/test/prompts.txt'
assert PROMPTS.exists(), f'Không thấy transcript VIVOS: {PROMPTS}'
references = {}
for line in PROMPTS.read_text(encoding='utf-8').splitlines():
    fields = line.strip().split(maxsplit=1)
    if len(fields) == 2:
        references[fields[0]] = fields[1]

def normalize_vietnamese(text):
    text = unicodedata.normalize('NFC', str(text)).lower()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

ASR_MAX_FILES = 20  # tăng sau khi chạy thử thành công
ASR_SNRS = [0, 5, 10]
ASR_DEVICE = -1  # CPU mặc định để tránh TensorFlow và PyTorch tranh GPU memory.
# Có thể đổi thành 0 nếu đã bật TensorFlow memory growth và GPU còn đủ bộ nhớ.
ASR_PARTIAL = EVAL_OUT / 'asr_utterances_partial.csv'
ASR_FINAL = EVAL_OUT / 'asr_utterances.csv'

asr_files = [p for p in test_files if p.stem in references]
if len(asr_files) > ASR_MAX_FILES:
    asr_files = random.Random(EVAL_SEED + 7).sample(asr_files, ASR_MAX_FILES)
assert asr_files, 'Không ghép được tên WAV với ID trong prompts.txt.'
print(f'PhoWhisper chạy trên {"GPU" if ASR_DEVICE >= 0 else "CPU"}; câu test={len(asr_files)}')

asr = pipeline('automatic-speech-recognition', model='vinai/PhoWhisper-small',
               device=ASR_DEVICE)

# Resume from Drive if an earlier run was interrupted.
if ASR_PARTIAL.exists():
    asr_df = pd.read_csv(ASR_PARTIAL)
    asr_rows = asr_df.to_dict('records')
    completed = {(str(r['file']), int(r['snr_db']), str(r['system'])) for r in asr_rows}
    print('Đang tiếp tục; kết quả ASR đã lưu:', len(completed))
else:
    asr_rows = []
    completed = set()

asr_rng = np.random.default_rng(EVAL_SEED + 7)
for i, file_path in enumerate(asr_files, 1):
    clean_raw = read_audio_16k(file_path)
    reference = references[file_path.stem]
    for snr_db in ASR_SNRS:
        keys = [(file_path.name, snr_db, system) for system in ['noisy', 'pretrained', 'finetuned']]
        if all(key in completed for key in keys):
            continue
        clean, noisy = mix_eval(clean_raw, make_noise(len(clean_raw), asr_rng), snr_db)
        candidates = {
            'noisy': noisy,
            'pretrained': enhance_one(baseline_model, noisy),
            'finetuned': enhance_one(finetuned_model, noisy),
        }
        for system, signal in candidates.items():
            key = (file_path.name, snr_db, system)
            if key in completed:
                continue
            prediction = asr({'raw': signal, 'sampling_rate': SR},
                generate_kwargs={'language': 'vietnamese', 'task': 'transcribe'})['text']
            asr_rows.append({'file': file_path.name, 'speaker': file_path.parent.name,
                'snr_db': snr_db, 'system': system, 'reference': reference,
                'hypothesis': prediction, 'reference_norm': normalize_vietnamese(reference),
                'hypothesis_norm': normalize_vietnamese(prediction)})
            completed.add(key)
        # Save after each utterance/SNR so a runtime interruption loses little work.
        pd.DataFrame(asr_rows).to_csv(ASR_PARTIAL, index=False)
    if i % 2 == 0 or i == len(asr_files):
        print(f'ASR {i}/{len(asr_files)} utterances; saved={len(completed)} predictions')

asr_results = pd.DataFrame(asr_rows)
asr_results.to_csv(ASR_FINAL, index=False)
asr_summary_rows = []
for (snr_db, system), group in asr_results.groupby(['snr_db', 'system']):
    refs = group.reference_norm.tolist()
    hyps = group.hypothesis_norm.tolist()
    asr_summary_rows.append({'snr_db': snr_db, 'system': system,
        'wer': jiwer.wer(refs, hyps), 'cer': jiwer.cer(refs, hyps),
        'utterances': len(group)})
asr_summary = pd.DataFrame(asr_summary_rows).sort_values(['snr_db', 'system'])
asr_summary.to_csv(EVAL_OUT / 'asr_summary.csv', index=False)
display(asr_summary.pivot(index='snr_db', columns='system', values='wer').round(4))
display(asr_summary.pivot(index='snr_db', columns='system', values='cer').round(4))
print('ASR transcript tables saved:', EVAL_OUT)


## 7. Biểu đồ so sánh và nghe thử

In [ ]:
# Paired comparisons from the same utterance and noise realization.
system_order = ['noisy', 'pretrained', 'finetuned']
plot_colors = {'noisy': '#7c8798', 'pretrained': '#e08b36', 'finetuned': '#2676c9'}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for system in system_order:
    part = summary[summary.system == system].sort_values('snr_db')
    axes[0].plot(part.snr_db, part.si_sdr_db, marker='o', linewidth=2,
                 label=system, color=plot_colors[system])
    axes[1].plot(part.snr_db, part.stoi, marker='o', linewidth=2,
                 label=system, color=plot_colors[system])
axes[0].set(title='SI-SDR theo SNR', xlabel='SNR đầu vào (dB)', ylabel='SI-SDR trung bình (dB)')
axes[1].set(title='STOI theo SNR', xlabel='SNR đầu vào (dB)', ylabel='STOI trung bình (0–1)')
for ax in axes:
    ax.grid(True, alpha=.25); ax.legend(frameon=False); ax.set_xticks(EVAL_SNRS)
fig.suptitle('VIVOS test: noisy vs DTLN pretrained vs fine-tuned')
fig.savefig(EVAL_OUT / 'metrics_by_snr.png', dpi=180, bbox_inches='tight')
plt.show()

# Paired gain over the noisy input, with bootstrap 95% confidence intervals.
paired = results.pivot(index=['file', 'speaker', 'snr_db'], columns='system', values='si_sdr_db').reset_index()
paired['pretrained_gain_db'] = paired['pretrained'] - paired['noisy']
paired['finetuned_gain_db'] = paired['finetuned'] - paired['noisy']
boot_rng = np.random.default_rng(EVAL_SEED)
fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
for col, label, color in [('pretrained_gain_db', 'pretrained − noisy', plot_colors['pretrained']),
                          ('finetuned_gain_db', 'fine-tuned − noisy', plot_colors['finetuned'])]:
    means, lows, highs = [], [], []
    for snr in EVAL_SNRS:
        values = paired.loc[paired.snr_db == snr, col].dropna().to_numpy()
        means.append(float(values.mean()))
        boot = np.array([boot_rng.choice(values, len(values), replace=True).mean() for _ in range(1000)])
        lows.append(float(np.quantile(boot, .025))); highs.append(float(np.quantile(boot, .975)))
    ax.plot(EVAL_SNRS, means, marker='o', linewidth=2, label=label, color=color)
    ax.fill_between(EVAL_SNRS, lows, highs, color=color, alpha=.15)
ax.axhline(0, color='gray', linewidth=1)
ax.set(title='SI-SDR gain so với input nhiễu (95% bootstrap CI)',
       xlabel='SNR đầu vào (dB)', ylabel='Cải thiện SI-SDR (dB)')
ax.set_xticks(EVAL_SNRS); ax.grid(True, alpha=.25); ax.legend(frameon=False)
fig.savefig(EVAL_OUT / 'si_sdr_gain_with_ci.png', dpi=180, bbox_inches='tight')
plt.show()

# Per-utterance paired distribution: positive values mean fine-tuned beats pretrained.
paired['fine_vs_pretrained_db'] = paired['finetuned'] - paired['pretrained']
box_data = [paired.loc[paired.snr_db == snr, 'fine_vs_pretrained_db'].dropna().to_numpy()
            for snr in EVAL_SNRS]
fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.boxplot(box_data, tick_labels=[str(x) for x in EVAL_SNRS], showfliers=False)
ax.axhline(0, color='gray', linewidth=1)
ax.set(title='Cải thiện SI-SDR từng câu: fine-tuned − pretrained',
       xlabel='SNR đầu vào (dB)', ylabel='Chênh lệch SI-SDR (dB)')
ax.grid(axis='y', alpha=.25)
fig.savefig(EVAL_OUT / 'paired_gain_distribution.png', dpi=180, bbox_inches='tight')
plt.show()

# Each dot is the same VIVOS utterance and noise draw; points above y=x favor fine-tuning.
snr_scatter = 0
same = paired[paired.snr_db == snr_scatter]
lo = min(same['pretrained'].min(), same['finetuned'].min())
hi = max(same['pretrained'].max(), same['finetuned'].max())
fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
ax.scatter(same['pretrained'], same['finetuned'], alpha=.65, color=plot_colors['finetuned'])
ax.plot([lo, hi], [lo, hi], linestyle='--', color='gray', linewidth=1)
ax.set(title='SI-SDR từng câu tại 0 dB', xlabel='Pretrained SI-SDR (dB)', ylabel='Fine-tuned SI-SDR (dB)')
ax.grid(True, alpha=.25)
fig.savefig(EVAL_OUT / 'paired_scatter_0db.png', dpi=180, bbox_inches='tight')
plt.show()

# Waveform and spectrogram at the same 0 dB test utterance.
if example is not None:
    duration = min(.8, len(example['clean']) / SR)
    n = max(1, int(duration * SR)); t = np.arange(n) / SR
    fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
    for ax, key, label in zip(axes,
        ['clean', 'noisy', 'pretrained', 'finetuned'],
        ['Clean reference', 'Noisy input', 'DTLN pretrained', 'DTLN fine-tuned']):
        ax.plot(t, example[key][:n], linewidth=.8, color=plot_colors.get(key, '#555555'))
        ax.set_ylabel(label); ax.grid(True, alpha=.2)
    axes[-1].set_xlabel('Thời gian (s)')
    fig.suptitle(f'Waveform 0 dB — {example["file"]}')
    fig.savefig(EVAL_OUT / 'waveform_0db_example.png', dpi=180, bbox_inches='tight')
    plt.show()

    fig, axes = plt.subplots(4, 1, figsize=(11, 9), constrained_layout=True)
    for ax, key, title in zip(axes,
        ['clean', 'noisy', 'pretrained', 'finetuned'],
        ['Clean reference', 'Noisy input (0 dB)', 'DTLN pretrained', 'DTLN fine-tuned']):
        f, t_spec, spec = spectrogram(example[key], fs=SR, nperseg=512, noverlap=384,
                                      scaling='spectrum', mode='magnitude')
        keep = f <= 8000
        ax.pcolormesh(t_spec, f[keep], 20*np.log10(spec[keep] + 1e-8), shading='auto', cmap='magma', vmin=-90, vmax=-25)
        ax.set(title=title, ylabel='Hz', ylim=(0, 8000))
    axes[-1].set_xlabel('Thời gian (s)')
    fig.suptitle(f'Spectrogram — {example["file"]}')
    fig.savefig(EVAL_OUT / 'spectrogram_0db_example.png', dpi=180, bbox_inches='tight')
    plt.show()
    display(Audio(example['noisy'], rate=SR))
    display(Audio(example['pretrained'], rate=SR))
    display(Audio(example['finetuned'], rate=SR))

# Optional downstream plot: shown after the PhoWhisper evaluation cell has run.
if 'asr_summary' in globals() and len(asr_summary):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    for system in system_order:
        part = asr_summary[asr_summary.system == system].sort_values('snr_db')
        axes[0].plot(part.snr_db, part.wer, marker='o', label=system, color=plot_colors[system])
        axes[1].plot(part.snr_db, part.cer, marker='o', label=system, color=plot_colors[system])
    axes[0].set(title='WER sau ASR', ylabel='WER (thấp hơn tốt hơn)')
    axes[1].set(title='CER sau ASR', ylabel='CER (thấp hơn tốt hơn)')
    for ax in axes:
        ax.set_xlabel('SNR đầu vào (dB)'); ax.grid(True, alpha=.25); ax.legend(frameon=False)
    fig.suptitle('PhoWhisper-small trên cùng câu test/noise')
    fig.savefig(EVAL_OUT / 'asr_wer_cer_by_snr.png', dpi=180, bbox_inches='tight')
    plt.show()

print('Các bảng và biểu đồ đã lưu ở:', EVAL_OUT)
